# Stage 2 · The whole course + lecture filter

In Stage 1 we built and understood every piece on **one** lecture. Here we reuse the same pieces on **all** the lectures.

The functions now live in `rag_core.py` (a "toolbox" file): the notebook only *uses* them, so it stays short and readable. Each function in `rag_core.py` keeps its `SAME` / `ADAPTED` / `NEW` comment.

In [ ]:
from rag_core import build_chunks_from_folder, load_or_compute_embeddings, search, answer, pages_label
from collections import Counter

## 1 · Read, clean and split every lecture

Same pipeline as Stage 1 (read → clean → split), repeated for each PDF in the folder. Cleaning is done **per lecture**, so each one finds its own header.

In [ ]:
chunks = build_chunks_from_folder("data/rag_papers")

print("Total chunks:", len(chunks))
for lecture, n in sorted(Counter(c["document"] for c in chunks).items()):
    print(f"  {lecture}: {n} chunks")

## 2 · Embeddings for the whole course

Same function as Stage 1, with a new cache name (`course`). The first time it calls Gemini for every chunk (a few minutes); after that it loads from disk.

In [ ]:
embeddings = load_or_compute_embeddings(chunks, name="course")
print("Shape:", embeddings.shape)

## 3 · Search with and without the lecture filter

The same question, asked to the whole course and then only to one lecture. Look at **which lectures** the results come from.

In [ ]:
question = "What is a DataFrame?"

print("=== Whole course ===")
for score, chunk in search(question, chunks, embeddings):
    print(f"{score:.3f} | {chunk['document']} {pages_label(chunk)} | {chunk['text'][:60]!r}")

print("\n=== Only lecture 3 (Pandas) ===")
for score, chunk in search(question, chunks, embeddings, lecture="lec3_slides"):
    print(f"{score:.3f} | {chunk['document']} {pages_label(chunk)} | {chunk['text'][:60]!r}")

## 4 · Answer

In [ ]:
text, used = answer("¿Qué es el overfitting y cómo se evita?", chunks, embeddings)
print(text)
print("\n--- Sources used ---")
for score, chunk in used:
    print(f"{score:.3f} | {chunk['document']} {pages_label(chunk)}")

In [ ]:
text, used = answer("¿Qué es el overfitting y cómo se evita?", chunks, embeddings, lecture="lec5_slides")
print(text)
print("\n--- Sources used ---")
for score, chunk in used:
    print(f"{score:.3f} | {chunk['document']} {pages_label(chunk)}")